In [ ]:
#Local Optima vs Global Optima in State-Space Search
def objective(x):
    # Function that we want to maximize
    return -(x - 3) ** 2 + 10


def find_local_optimum(start):
    current = start

    while True:
        # Check neighboring states
        left = current - 1
        right = current + 1

        best = current

        if objective(left) > objective(best):
            best = left

        if objective(right) > objective(best):
            best = right

        # If no neighbor is better, we reached a local optimum
        if best == current:
            break

        current = best

    return current, objective(current)


# Starting state
start = 0

x, value = find_local_optimum(start)

print("Starting State:", start)
print("Local Optimum:", x)
print("Value:", value)

# Find global optimum by checking the complete state space
global_x = max(range(-10, 11), key=objective)

print("Global Optimum:", global_x)
print("Global Maximum Value:", objective(global_x))

In [ ]:
#Exploration vs Exploitation Trade-off
import random

states = [10, 15, 20, 25, 30, 35, 40, 45, 50]

current = random.choice(states)

print("Starting State:", current)

for i in range(10):

    # EXPLORATION
    if random.random() < 0.3:
        current = random.choice(states)
        print("Exploring new state:", current)

    # EXPLOITATION
    else:
        better_states = [x for x in states if x > current]

        if better_states:
            current = max(better_states)

        print("Exploiting current solution:", current)

print("Final State:", current)

In [ ]:
#Hill Climbing Search — Eight Queens Problem
import random


# Calculate number of attacking queen pairs
def calculate_conflicts(board):
    conflicts = 0

    for i in range(8):
        for j in range(i + 1, 8):

            # Same row
            if board[i] == board[j]:
                conflicts += 1

            # Same diagonal
            if abs(board[i] - board[j]) == abs(i - j):
                conflicts += 1

    return conflicts


# Print chessboard
def print_board(board):
    for row in range(8):
        for col in range(8):
            if board[col] == row:
                print("Q", end=" ")
            else:
                print(".", end=" ")
        print()


def hill_climbing():

    # Random initial state
    board = [random.randint(0, 7) for _ in range(8)]

    while True:

        current_conflicts = calculate_conflicts(board)

        # Solution found
        if current_conflicts == 0:
            return board

        best_board = board[:]
        best_conflicts = current_conflicts

        # Try moving every queen
        for col in range(8):

            original_row = board[col]

            for row in range(8):

                if row == original_row:
                    continue

                board[col] = row

                conflicts = calculate_conflicts(board)

                if conflicts < best_conflicts:
                    best_conflicts = conflicts
                    best_board = board[:]

            board[col] = original_row

        # No better state exists
        if best_conflicts >= current_conflicts:
            print("Hill Climbing got stuck at a local optimum.")
            return None

        board = best_board


# Run the algorithm
solution = hill_climbing()

if solution:
    print("Solution Found:")
    print("State:", solution)
    print("Conflicts:", calculate_conflicts(solution))

    print("\nChess Board:")
    print_board(solution)
else:
    print("No solution found.")

In [ ]:
#Simulated Annealing — Eight Queens
import random
import math


def calculate_conflicts(board):
    conflicts = 0

    for i in range(8):
        for j in range(i + 1, 8):

            if board[i] == board[j]:
                conflicts += 1

            if abs(board[i] - board[j]) == abs(i - j):
                conflicts += 1

    return conflicts


def simulated_annealing():

    # Random starting state
    board = [random.randint(0, 7) for _ in range(8)]

    temperature = 100

    while temperature > 0.01:

        current_conflicts = calculate_conflicts(board)

        if current_conflicts == 0:
            return board

        # Create a neighboring state
        new_board = board[:]

        column = random.randint(0, 7)
        new_board[column] = random.randint(0, 7)

        new_conflicts = calculate_conflicts(new_board)

        # Improvement
        if new_conflicts < current_conflicts:
            board = new_board

        else:
            # Difference in quality
            difference = new_conflicts - current_conflicts

            # Probability of accepting worse solution
            probability = math.exp(-difference / temperature)

            if random.random() < probability:
                board = new_board

        # Cool down
        temperature *= 0.95

    return board


solution = simulated_annealing()

print("Solution:", solution)
print("Conflicts:", calculate_conflicts(solution))

In [ ]:
#Simulated Annealing — Travelling Salesman Problem(Case 1)
import random
import math


# Distance between two cities
def distance(city1, city2):
    x1, y1 = city1
    x2, y2 = city2

    return math.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)


# Calculate total route distance
def route_distance(route, cities):

    total = 0

    for i in range(len(route)):
        current = route[i]
        next_city = route[(i + 1) % len(route)]

        total += distance(
            cities[current],
            cities[next_city]
        )

    return total


def simulated_annealing(cities):

    # Random route
    route = list(range(len(cities)))
    random.shuffle(route)

    temperature = 1000

    while temperature > 0.01:

        current_distance = route_distance(route, cities)

        # Create a neighboring route
        new_route = route[:]

        i, j = random.sample(range(len(cities)), 2)

        # Swap two cities
        new_route[i], new_route[j] = new_route[j], new_route[i]

        new_distance = route_distance(new_route, cities)

        # Better route
        if new_distance < current_distance:
            route = new_route

        else:
            difference = new_distance - current_distance

            probability = math.exp(
                -difference / temperature
            )

            if random.random() < probability:
                route = new_route

        temperature *= 0.995

    return route


cities = [
    (0, 0),
    (2, 5),
    (5, 2),
    (7, 7),
    (1, 8)
]

best_route = simulated_annealing(cities)

print("Best Route:", best_route)

print(
    "Total Distance:",
    route_distance(best_route, cities)
)

In [ ]:
#Simulated Annealing — Elevator Optimization(Case 2)
import random
import math


# Passenger requests
# (source floor, destination floor)
requests = [
    (1, 8),
    (2, 5),
    (7, 3),
    (4, 9),
    (1, 6),
    (8, 2)
]

number_of_elevators = 3


# Calculate total travel cost
def calculate_cost(assignments):

    total_cost = 0

    for i in range(len(requests)):

        source, destination = requests[i]

        # Travel distance
        travel = abs(destination - source)

        # Elevator assignment
        elevator = assignments[i]

        # Simple load penalty
        load = assignments.count(elevator)

        total_cost += travel + (load * 2)

    return total_cost


def simulated_annealing():

    # Random initial assignment
    current = [
        random.randint(0, number_of_elevators - 1)
        for _ in requests
    ]

    temperature = 100

    while temperature > 0.01:

        current_cost = calculate_cost(current)

        # Generate neighboring solution
        new_solution = current[:]

        request = random.randint(0, len(requests) - 1)

        new_solution[request] = random.randint(
            0,
            number_of_elevators - 1
        )

        new_cost = calculate_cost(new_solution)

        # Better solution
        if new_cost < current_cost:
            current = new_solution

        else:

            difference = new_cost - current_cost

            probability = math.exp(
                -difference / temperature
            )

            if random.random() < probability:
                current = new_solution

        # Cooling
        temperature *= 0.95

    return current


best_assignment = simulated_annealing()

print("Passenger Requests:")
print(requests)

print("\nBest Elevator Assignment:")

for i in range(len(requests)):
    print(
        "Request",
        i + 1,
        "-> Elevator",
        best_assignment[i] + 1
    )

print(
    "\nTotal Cost:",
    calculate_cost(best_assignment)
)

In [ ]:
#VLSI Layout — Simulated Annealing(Case 3)
import random
import math


components = ["A", "B", "C", "D", "E"]

connections = [
    ("A", "B"),
    ("A", "C"),
    ("B", "D"),
    ("C", "E"),
    ("D", "E")
]


def calculate_cost(layout):

    cost = 0

    for a, b in connections:

        x1, y1 = layout[a]
        x2, y2 = layout[b]

        # Manhattan distance
        cost += abs(x1 - x2) + abs(y1 - y2)

    return cost


def simulated_annealing():

    # Random initial positions
    layout = {}

    positions = [
        (0, 0),
        (0, 1),
        (1, 0),
        (1, 1),
        (2, 0)
    ]

    random.shuffle(positions)

    for i in range(len(components)):
        layout[components[i]] = positions[i]

    temperature = 100

    while temperature > 0.01:

        current_cost = calculate_cost(layout)

        new_layout = layout.copy()

        # Select two components
        a, b = random.sample(components, 2)

        # Swap their positions
        new_layout[a], new_layout[b] = (
            new_layout[b],
            new_layout[a]
        )

        new_cost = calculate_cost(new_layout)

        if new_cost < current_cost:
            layout = new_layout

        else:

            difference = new_cost - current_cost

            probability = math.exp(
                -difference / temperature
            )

            if random.random() < probability:
                layout = new_layout

        temperature *= 0.95

    return layout


best_layout = simulated_annealing()

print("Best VLSI Layout:")

for component, position in best_layout.items():
    print(component, "->", position)

print("\nTotal Wire Length:",
      calculate_cost(best_layout))